Clustering: segmentation des clients

Cas: KPMG_Customer_Demography

Data Analyst: Prince Obenga

IMPORTATION DES DONNEES

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/KPMG_Customer_Demography.csv')
df.head()

NETTOYAGE

In [ ]:
df = df.drop(columns=['customer_id','First_Name', 'Last_Name', 'DOB', 'Postcode', 'Country', 'deceased_indicator', 'Address'], errors='ignore')
df.head()

stats descriptives

In [ ]:
df.describe()

In [ ]:
df['State'].value_counts().sort_index()

In [ ]:
df['tenure'].value_counts().sort_index()

analyse univariée et bivariée

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns




In [ ]:
x = df['gender']



plt.figure(figsize=(6, 3.5))
sns.countplot(data=df, x='gender', order=x.value_counts().index)
plt.title('Répartition par genre')
plt.show()

In [ ]:
x = df['wealth_segment']


plt.figure(figsize=(6, 3.5))
sns.countplot(data=df, x='wealth_segment', order=x.value_counts().index)
plt.title('Répartition par segment de richesse')
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df['past_3_years_bike_related_purchases'])
plt.title('Distribution des achats de vélos sur 3 ans')
plt.xlabel('Nombre d\'achats')
plt.ylabel('Effectif')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='property_valuation')
plt.title('Effectifs par valeur de propriété')
plt.xlabel('property_valuation')
plt.ylabel('Effectif')
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.regplot(data=df, x='AGE', y='past_3_years_bike_related_purchases',
            scatter_kws={'alpha': 0.3}, line_kws={'color': 'red'})
plt.title('Achats de vélos selon l\'âge')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.lineplot(data=df, x='tenure', y='past_3_years_bike_related_purchases')
plt.title('Achats de vélos moyens selon l\'ancienneté')
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.regplot(data=df, x='AGE', y='tenure',
            scatter_kws={'alpha': 0.3}, line_kws={'color': 'red'})
plt.title('Ancienneté selon l\'âge')
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=df, x='gender', y='past_3_years_bike_related_purchases', errorbar=None)
plt.title('Achats de vélos moyens selon le genre')
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=df, x='wealth_segment', y='past_3_years_bike_related_purchases', errorbar=None)
plt.title('Achats de vélos moyens selon le segment de richesse')
plt.show()

**FUTUR** **ENGINEERING**

In [ ]:
df['Age_group'] = pd.cut(df['AGE'],
                         bins=[0, 34, 54, 120],
                         labels=['jeune', 'adulte', 'senior'])
df['Age_group'].value_counts().sort_index()

In [ ]:
df['Purchase_intensity'] = (df['past_3_years_bike_related_purchases'] / df['tenure']).round(2)
df['Purchase_intensity'].value_counts().sort_index()

In [ ]:
df['Wealth_score'] = (df['past_3_years_bike_related_purchases'] * df['property_valuation'])
df['Wealth_score'].value_counts()


In [ ]:
df['Annual_purchase_frequency'] = (df['past_3_years_bike_related_purchases'] * 3)
df['Annual_purchase_frequency'].value_counts()

In [ ]:
df['Buyer_potential'] = pd.cut(df['tenure'],
                         bins=[0, 3, 7, 20],
                         labels=['Occasionnel', 'Regulier', 'Passionné'])
df['Buyer_potential'].value_counts().sort_index()

LA METHODE DES K-MEANS

In [ ]:
from sklearn import datasets
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from yellowbrick.cluster import SilhouetteVisualizer

# Perform one-hot encoding
df_encoded = pd.get_dummies(df, dtype = int)
X=df_encoded


In [ ]:




fig, ax = plt.subplots(3, 2, figsize=(15,8))
for i in [2, 3, 4, 5, 6, 7]:
    '''
    Create KMeans instance for different number of clusters
    '''
    km = KMeans(n_clusters=i, init='k-means++', n_init=10, max_iter=100, random_state=42)
    q, mod = divmod(i, 2)
    '''
    Create SilhouetteVisualizer instance with KMeans instance
    Fit the visualizer
    '''
    visualizer = SilhouetteVisualizer(km, colors='yellowbrick', ax=ax[q-1][mod])
    visualizer.fit(X)

In [ ]:
# Calculate distortions for different numbers of clusters
distortions = []
K = range(1, 10)  # Test clusters from 1 to 9
for k in K:
    kmeanModel = KMeans(n_clusters=k)
    kmeanModel.fit(X)
    distortions.append(kmeanModel.inertia_)

# Plot the elbow method graph
plt.figure(figsize=(16, 8))
plt.plot(K, distortions, 'bx-')
plt.xlabel('k')
plt.ylabel('Distortion')
plt.title('The Elbow Method showing the optimal k')
plt.show()

In [ ]:
from sklearn.decomposition import PCA


kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

# Réduction de dimensionnalité avec PCA pour visualisation
pca = PCA(n_components=2)
data_2d = pca.fit_transform(X)

# Visualisation des clusters
plt.scatter(data_2d[:, 0], data_2d[:, 1], c=labels, cmap='viridis', marker='o')
plt.scatter(pca.transform(kmeans.cluster_centers_)[:, 0], pca.transform(kmeans.cluster_centers_)[:, 1], color='red', marker='x')
plt.title('Clusters K-Means avec PCA')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.show()

**DESCRIPTION POUR ANALYSE**

In [ ]:
import pandas as pd

# convertir X en DataFrame
columns = ['Wealth_score', 'Annual_purchase_frequency','property_valuation','AGE']
df_clustered = pd.DataFrame(X, columns=columns)

# ajouter les labels des clusters au DataFrame
df_clustered['cluster_label'] = labels

# afficher
print(df_clustered.head())

In [ ]:
cluster_description = df_clustered.groupby('cluster_label').describe()
print(cluster_description)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Utiliser les vraies colonnes numériques disponibles dans df_clustered
feature_columns = ['Wealth_score', 'Annual_purchase_frequency','property_valuation','AGE']

# Créer les boxplots pour chaque caractéristique numérique en groupant par cluster
plt.figure(figsize=(12, 5))
for i, col in enumerate(feature_columns):
    plt.subplot(2, 2, i + 1) # Arrange plots in a 2x2 grid
    sns.boxplot(x='cluster_label', y=col, data=df_clustered)
    plt.title(f'Boxplot of {col.replace("_", " ").title()} by Cluster')
    plt.xlabel('Cluster')
    plt.ylabel(col.replace("_", " ").title())

plt.tight_layout()
plt.show()

**Analyse des resultats et insight**

Le K-Means a séparé les clients en trois groupes, surtout selon leur score de richesse et leur fréquence d’achat, qui évoluent ensemble.

Le cluster 0 (854 clients, environ 22 %) regroupe les clients premium : le plus riche (Wealth_score d’environ 780) et le plus actif (fréquence d’environ 249 achats). Le cluster 2 (1 337 clients, environ 34 %) est intermédiaire, avec un Wealth_score d’environ 420 et une fréquence d’environ 170. Le cluster 1 (1 717 clients, environ 44 %) est le plus nombreux et celui de plus faible valeur, avec un Wealth_score d’environ 125 et une fréquence d’environ 79.

Les boxplots confirment cette séparation, surtout pour Wealth_score, où les boîtes ne se chevauchent presque pas. L’âge est le même dans les trois groupes (médiane de 45 ans) et la property_valuation discrimine peu, donc ces deux variables n’expliquent pas la segmentation.

L'action a preconiser pour l'équpe marketing serait de preparer des offres distincts pour chaque segement.
Pour le premier segment que j'appelle segment premium, je preconise un servive vip ou premium adapté à ce type de clients, ainsi cela aura un effet fidelisateur et de valorisqtion de ceux ci car ce sont les meilleurs clients, en tout cas ceux qui rapportent le plus.

Ensuite on a le segment intermediaire,qui represente une bonne partie de la clientele  avec une bonne frequence d'achat, à ceux ci je preconise une strategie orientée à leurs besoins.

Quant au dernier segment(le plus grand) qui represente pres de la moitié des clients, il sied de constater que ce segment a une particularité, achat occasionnel, adapté à leurs moyen. On a ici des clients entrant, dormant et c'est ici qu'on a les plus risque de churn. A ce segment je preconise des offres de bienvenu, un systeme d'accompagnement des nouveaux clients, des ffres cibles pour inciter à l'achat et surtout des remises couplés aux cartes de fidelité pour touché favorablement ce segment.